# Spatial vs pooled decoding of the rendered count
Is the count carried by the **spatial layout** of up0 self-attention rather than a global summary? We decode the rendered count three ways, all with **5-fold cross-validated ridge $R^2$** (stable at small $n$; ~0 when uninformative, not wildly negative):
1. **pooled** channel vector (mean over tokens) - global/linear baseline;
2. **grid** raw-energy at resolution $g$ (spatial, linear);
3. **peak-count** - number of saliency peaks (spatial, nonlinear readout).

**Runtime:** GPU (~15 min).

In [ ]:
import os
if not os.path.exists('src'):
    !git clone https://github.com/serinaqin/T2I-Count-Anomaly.git
    %cd T2I-Count-Anomaly
else:
    !git pull   # get latest src fixes (NOTE: still Restart Runtime to reload imported modules)
!pip install -q -r requirements.txt
!pip install -q pytest groundingdino-py

In [ ]:
import sys; sys.path.insert(0, '.')
import numpy as np, pandas as pd, os, yaml
import matplotlib.pyplot as plt
from src.prompts import generate_grid
from src.pipeline import (load_sdxl, catalog_attention_sites,
                          select_probe_sites, generate_and_capture,
                          raw_reducer, pool_activation)
from src.spatial import featuremap_saliency, grid_pool_2d, count_peaks
from src.probes import cv_r2
from src.detector import Detector
from src.scoring import count_from_detections
from src.config import load_config

In [ ]:
cfg = load_config('configs/exp_spatialprobe.yaml')
raw = yaml.safe_load(open('configs/exp_spatialprobe.yaml'))
step = raw['capture_step']; grid_sizes = raw['grid_sizes']
block, attn = raw['patch_block'], raw['patch_attn']
pipe = load_sdxl(); det = Detector()
sites = [s for s in select_probe_sites(catalog_attention_sites(pipe.unet))
         if block in s and s.endswith(attn)]
def cnt(img, obj):
    return count_from_detections(det.detect(img, [obj]), obj, cfg.score_threshold)
print('sites:', sites, '| capture step', step)

In [ ]:
# Capture RAW activations; derive both a pooled channel vector and a raw
# (non-normalized) energy map per image, averaged over the 3 up0 attn1 sites.
grid = generate_grid(cfg.counts, cfg.objects, cfg.seeds)
pooled_list, sal_list, rows = [], [], []
for i, p in enumerate(grid):
    img, snaps = generate_and_capture(pipe, p.text, p.seed, sites, [step],
                                      cfg.num_inference_steps, reducer=raw_reducer)
    acts = [snaps[step][s] for s in sites if s in snaps.get(step, {})]
    pooled_list.append(np.mean([pool_activation(a) for a in acts], axis=0))          # (C,)
    sal_list.append(np.mean([featuremap_saliency(a, normalize=False) for a in acts], axis=0))  # (H,W) raw
    rows.append({'obj': p.obj, 'count': p.count, 'seed': p.seed, 'rendered': cnt(img, p.obj)})
    if (i + 1) % 20 == 0: print(f'{i+1}/{len(grid)}')
df = pd.DataFrame(rows)
os.makedirs('results', exist_ok=True)
df.to_csv('results/exp_spatialprobe_counts.csv', index=False)
print('captured', len(grid), 'maps of shape', sal_list[0].shape)

In [ ]:
# Cross-validated R^2 of decoding the rendered count each way.
y = df['rendered'].to_numpy(float)
pooled_r2 = cv_r2(np.array(pooled_list), y)                       # global channel probe
grid_r2 = {g: cv_r2(np.array([grid_pool_2d(s, g) for s in sal_list]), y) for g in grid_sizes}
pc = np.array([count_peaks(s, thresh_rel=0.5) for s in sal_list])  # nonlinear spatial readout
peak_r = float(np.corrcoef(pc, y)[0, 1]); peak_r2 = peak_r ** 2
print(f'pooled channel vector      cv_R2 = {pooled_r2:.3f}')
for g in grid_sizes:
    print(f'grid raw-energy g={g}        cv_R2 = {grid_r2[g]:.3f}')
print(f'peak-count readout          r = {peak_r:.3f}  (r^2 = {peak_r2:.3f})')
pd.DataFrame({'method': ['pooled(C)'] + [f'grid g={g}' for g in grid_sizes] + ['peak-count(r^2)'],
              'r2': [pooled_r2] + [grid_r2[g] for g in grid_sizes] + [peak_r2]}
             ).to_csv('results/exp_spatialprobe_r2.csv', index=False)

In [ ]:
methods = ['pooled(C)'] + [f'grid g={g}' for g in grid_sizes] + ['peak-count']
vals = [pooled_r2] + [grid_r2[g] for g in grid_sizes] + [peak_r2]
cols = ['gray'] + ['C0'] * len(grid_sizes) + ['C2']
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(methods, vals, color=cols)
ax.axhline(0, color='k', lw=0.6)
ax.set_ylabel('rendered-count decodability (R^2)')
ax.set_title('Spatial readouts vs a global pooled probe')
plt.xticks(rotation=20, ha='right'); plt.tight_layout()
plt.savefig('results/exp_spatialprobe_r2.png', dpi=110, bbox_inches='tight'); plt.show()

## How to read this
- **Spatial readouts (grid at g=2/4, and especially peak-count) clearly exceed the pooled channel probe** = the rendered count is carried by the *spatial layout* of up-block self-attention, more than by any global summary. That is the positive demonstration the earlier (broken) probe could not give, and it supports the object-layout framing.
- **peak-count $r^2$ highest** = the count is a *nonlinear, instance-multiplicity* property (a counting readout beats any linear probe) - consistent with a slot/layout code rather than a linear magnitude.
- **All comparable / pooled not beaten** = the count is not specifically spatial in this readout; report honestly and lean on the causal patch results instead.
- (Fixed vs the earlier version: raw energy instead of min-max saliency, and cross-validated ridge instead of a single unregularized split, which had produced spurious negative $R^2$.)